# Module 15: How Wrong Is the Forecast?

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Every forecast comes with an error measure, and the popular one is the worst
choice for public safety data.

This module covers which measure to use, why one holdout is not enough, and how
to check whether a prediction interval means what it says.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")


def split(s, end="2024-12", horizon=12):
    """Train on everything up to `end`, test on the next `horizon` months."""
    train = s.loc[:end]
    test = s.loc[pd.Timestamp(end) + pd.offsets.MonthBegin(1):][:horizon]
    return train, test


def mae(actual, pred):
    return float(np.mean(np.abs(np.asarray(actual, float) - np.asarray(pred, float))))

from statsmodels.tsa.holtwinters import ExponentialSmoothing

grandview = series("A012")

## 2. Four measures

| Measure | What it is | Units | Reads well when |
|---|---|---|---|
| **MAE** | average absolute error | incidents | you want a number people understand |
| **RMSE** | square root of the average squared error | incidents | large misses matter more than small ones |
| **MAPE** | average absolute error as a percent of the actual | percent | actuals are comfortably above zero |
| **MASE** | MAE divided by the MAE a naive forecast would get | a ratio | comparing across series of different sizes |

MASE is the one to default to. Below 1 means better than the naive benchmark;
above 1 means worse. It is comparable across agencies of any size, which none
of the others are.

In [ ]:
def scores(train, test, pred):
    a, p = np.asarray(test, float), np.asarray(pred, float)
    naive_scale = np.mean(np.abs(train.values[12:] - train.values[:-12]))
    with np.errstate(divide="ignore", invalid="ignore"):
        ape = np.abs((a - p) / a)
    return pd.Series({
        "MAE": np.mean(np.abs(a - p)),
        "RMSE": np.sqrt(np.mean((a - p) ** 2)),
        "MAPE": 100 * np.mean(ape),
        "MASE": np.mean(np.abs(a - p)) / naive_scale,
    })

In [ ]:
train, test = split(grandview, end="2024-12")
hw = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12,
                          initialization_method="estimated").fit()

pd.DataFrame({
    "Holt Winters": scores(train, test, hw.forecast(12)),
    "same month last year": scores(train, test, train.iloc[-12:].values),
}).round(2)

## 3. Why MAPE fails on public safety data

**It cannot be computed when the actual is zero.** Dividing by zero is not a
large percentage, it is undefined, and small agencies have zero months
routinely.

In [ ]:
elk_train, elk_test = split(series("A006"), end="2024-12")
pred = np.repeat(elk_train.iloc[-12:].mean(), 12)

print("Orrindale 2025 actual:", [int(x) for x in elk_test.values])
print(f"months at zero: {int((elk_test == 0).sum())} of 12")
print()
print(scores(elk_train, elk_test, pred).round(3).to_string())

MAE and MASE report perfectly usable numbers. MAPE reports `inf`, and software
that silently drops the zero months instead would report a number computed from
the seven months that happened not to be zero.

**It is also asymmetric.** The actual sits in the denominator, so the same
absolute miss is punished differently depending on which way it went.

In [ ]:
demo = pd.DataFrame([
    {"actual": 5,  "forecast": 10, "absolute error": 5, "APE, percent": 100},
    {"actual": 15, "forecast": 10, "absolute error": 5, "APE, percent": 33},
    {"actual": 10, "forecast": 5,  "absolute error": 5, "APE, percent": 50},
    {"actual": 10, "forecast": 15, "absolute error": 5, "APE, percent": 50},
])
demo

Every row is an absolute miss of five incidents. MAPE scores them at 100, 33
and 50 percent. A method that systematically over forecasts quiet months is
punished hardest, which is the opposite of what a public safety user usually
wants: **under forecasting a busy month is the more costly error.**

## 4. One holdout is one number

Everything so far rests on a single split at the end of 2024. Move the split
and the verdict moves with it.

In [ ]:
rows = []
for end in ["2022-12", "2023-06", "2023-12", "2024-06", "2024-12", "2025-04"]:
    tr, te = split(grandview, end)
    if len(te) < 12:
        continue
    f = ExponentialSmoothing(tr, trend="add", seasonal="add", seasonal_periods=12,
                             initialization_method="estimated").fit().forecast(12)
    rows.append({
        "train ends": end,
        "months of training": len(tr),
        "Holt Winters": round(mae(te.values, f.values), 2),
        "same month last year": round(mae(te.values, tr.iloc[-12:].values), 2),
    })

roll = pd.DataFrame(rows).set_index("train ends")
roll["improvement, percent"] = (100 * (1 - roll["Holt Winters"]
                                       / roll["same month last year"])).round(0)
roll

Two things fall out, and neither is visible from a single split.

**The usual holdout is the most flattering one.** Ending at December 2024 gives
the lowest error of the six. Reporting that number alone overstates how well
the method does.

**With four years of training data the model loses to the baseline.** At the
first origin, 48 months, Holt Winters is worse than using the same month last
year. The advantage appears only once there are five or six years to estimate
the seasonal shape from. A method is not good or bad on its own; it is good or
bad **given how much data you have**.

In [ ]:
print(f"Holt Winters error across the six origins: "
      f"{roll['Holt Winters'].min():.1f} to {roll['Holt Winters'].max():.1f}")
print(f"average across origins: {roll['Holt Winters'].mean():.1f}")
print(f"the single usual holdout reports: {roll.loc['2024-12', 'Holt Winters']:.1f}")

## 5. Does the interval mean what it says?

A 95 percent interval should contain the truth about 95 times in 100. Check it
rather than assuming it.

In [ ]:
covered = []
for end in ["2022-12", "2023-06", "2023-12", "2024-06", "2024-12"]:
    tr, te = split(grandview, end)
    f = ExponentialSmoothing(tr, trend="add", seasonal="add", seasonal_periods=12,
                             initialization_method="estimated").fit()
    p = f.forecast(12)
    sd = float(np.std(f.resid, ddof=1))
    covered.append(((te.values >= p.values - 1.96 * sd)
                    & (te.values <= p.values + 1.96 * sd)))

hits = np.concatenate(covered)
print(f"months checked: {len(hits)}")
print(f"inside the 95 percent interval: {hits.sum()} ({100 * hits.mean():.0f} percent)")

Close to nominal, and slightly under, which is the usual direction. The
interval is built from the spread of the model's own fitted errors and
therefore assumes the estimated parameters are correct. They are not, so real
coverage is normally a little worse than advertised. Reporting an interval
without ever checking its coverage is a claim nobody has tested.

## 6. A reusable evaluation

In [ ]:
def rolling_origin(s, origins, horizon=12, model=None):
    """Score a forecasting function at several starting points."""
    if model is None:
        def model(tr, h):
            return ExponentialSmoothing(tr, trend="add", seasonal="add",
                                        seasonal_periods=12,
                                        initialization_method="estimated"
                                        ).fit().forecast(h).values
    out = []
    for end in origins:
        tr, te = split(s, end, horizon)
        if len(te) < horizon or len(tr) < 36:
            continue
        pred = model(tr, horizon)
        scale = np.mean(np.abs(tr.values[12:] - tr.values[:-12]))
        out.append({"train ends": end, "MAE": mae(te.values, pred),
                    "MASE": mae(te.values, pred) / scale})
    return pd.DataFrame(out).set_index("train ends")

In [ ]:
rolling_origin(grandview, ["2023-06", "2023-12", "2024-06", "2024-12"]).round(2)

## 7. What to carry away

- **Report MAE for people and MASE for comparison.** Never MAPE on count data.
- **Score at several origins**, not one, and report the range.
- **A method's verdict depends on how much data it had.**
- **Check interval coverage.** An untested interval is an assertion.
- **Always score the baseline alongside**, from
  [Module 13](Module_13_Baseline_Forecasts.ipynb).

## Exercise

Score Holt Winters against the seasonal naive baseline for Tarnbridge at
several origins. Does it beat the baseline consistently?

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A002"

if AGENCY:
    s = series(AGENCY)
    origins = ["2022-12", "2023-06", "2023-12", "2024-06", "2024-12"]
    rows = []
    for end in origins:
        tr, te = split(s, end)
        if len(te) < 12:
            continue
        f = ExponentialSmoothing(tr, trend="add", seasonal="add", seasonal_periods=12,
                                 initialization_method="estimated").fit().forecast(12)
        rows.append({"train ends": end,
                     "Holt Winters": round(mae(te.values, f.values), 2),
                     "baseline": round(mae(te.values, tr.iloc[-12:].values), 2)})
    out = pd.DataFrame(rows).set_index("train ends")
    out["Holt Winters wins"] = out["Holt Winters"] < out["baseline"]
    print(out.to_string())
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
```

Holt Winters wins at some origins and loses at others. That is the honest
result for an agency of this size, and it is the reason to score at several
starting points rather than one.

Tarnbridge averages about 28 incidents a month against Ashfell's 100, so a
larger share of its month to month movement is counting noise
([Module 4](Module_04_Why_Small_Agencies_Look_Volatile.ipynb) once more) and
there is less stable structure for the model to find. When a method wins
sometimes and loses sometimes, the defensible report is that it does not
reliably beat the free answer, and the free answer is easier to maintain and
easier to explain.

</details>

---

**Next:** [Module 16, Did Something Change](Module_16_Did_Something_Change.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*